In [ ]:
import torch
import timm
import torchvision.transforms as transforms
from torchvision.datasets import ImageNet
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from utils import InputHook, kernel_alignment

In [ ]:
def get_imagenet_batch(path, batch_size, device):
    """Loads a single batch from the ImageNet validation set."""
    transform = transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])
    dataset = ImageNet(path, transform=transform, split='val')
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
    
    # Return just the first batch of images
    images, _ = next(iter(loader))
    return images.to(device)

In [ ]:
# --- Configuration ---
MODEL_NAMES = [
    'resnet18', 
    'resnet34', 
    'resnet50',
    'resnet101',
]
IMAGENET_VAL_PATH = '' # Replace with ImageNet Directory
BATCH_SIZE = 64
SOFT_BETA = 0.8
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
print(f"Loading {BATCH_SIZE} images from ImageNet...")
batch = get_imagenet_batch(IMAGENET_VAL_PATH, BATCH_SIZE, DEVICE)

reference_model=timm.create_model('resnet152', pretrained=True, num_classes=1000).to(DEVICE)
reference_model.eval()

pretrained_alignments={}
random_alignments={}

for beta in [SOFT_BETA, 1.0]:
    hook = InputHook(reference_model, beta=beta)
    with torch.no_grad():
        hook(batch)
    reference_kernel=hook.vq_kernel
    for pretrained in [True, False]:
        kernels = []
    
        for model_name in MODEL_NAMES:
            print(f"Processing {'Pretrained' if pretrained else 'Random'} {model_name}...for beta={beta} ")
            
            # Load model with timm
            model = timm.create_model(model_name, pretrained=pretrained, num_classes=1000).to(DEVICE)
            model.eval()
            
            # Initialize hook (it auto-registers for ReLUs per your vqk.py)
            hook = InputHook(model, beta=beta)
            
            with torch.no_grad():
                hook(batch)
            
            # Compute the integrated kernel across all ReLU layers
            k = hook.vq_kernel
            kernels.append(k)
            # Cleanup to free memory
            hook.remove()
            del model
            torch.cuda.empty_cache()

        alignment_values = []
        frobenius_norms = []
        
        print("Computing alignment scores...")
        for k in kernels:
            alignment_values.append(kernel_alignment(k, reference_kernel).item())

        if pretrained:
            pretrained_alignments[beta]=alignment_values
        else:
            random_alignments[beta]=alignment_values

In [ ]:
N=len(MODEL_NAMES)

fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(5, 4), dpi=200)

l1, = ax.plot(range(N), pretrained_alignments[SOFT_BETA], color=plt.cm.winter(0.25),
              marker='o', label='SoftVQ')
l2, = ax.plot(range(N), pretrained_alignments[1.0], color=plt.cm.autumn(0.25),
              marker='o', label='HardVQ')

ax.plot(range(N), random_alignments[SOFT_BETA], color=plt.cm.winter(0.25),
        marker='o', linestyle='--')
ax.plot(range(N), random_alignments[1.0], color=plt.cm.autumn(0.25),
        marker='o', linestyle='--')

ax.grid(linestyle='--', color='grey', alpha=0.25)
ax.set_ylabel('Kernel Alignment')
ax.set_xticks(range(N))
ax.set_xticklabels(MODEL_NAMES)


legend1 = ax.legend(loc='center left', title='Method')

line_trained = Line2D([0], [0], color='black', linestyle='-',
                      marker='o', label='Trained')
line_random = Line2D([0], [0], color='black', linestyle='--',
                     marker='o', label='Random')

legend2 = ax.legend(handles=[line_trained, line_random],
                    loc='center right', title='Weights')

ax.add_artist(legend1)

plt.show()
